In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

# --- 1. Configuration ---
# Point this to the file that contains ALL your patients and their true TMB values
project_dir = Path.cwd().resolve().parent.parent.parent
MASTER_DATA_PATH = Path(project_dir/"data/raw/coad_tmb.parquet")
MASTER_RNA_PATH = Path("/media/maro/Mom0-0/Datasets/TCGA/TCGA-STAD-transcriptional/gdc_sample_sheet.2026-05-14.tsv")
OUTPUT_PATH = Path(project_dir/"data/processed/master_patient_splits.csv")

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42 # Locks the randomness so everyone generates the exact same file

print("Loading master dataset...")
df = pd.read_parquet(MASTER_DATA_PATH)
rna_df = pd.read_csv(MASTER_RNA_PATH, sep="\t")
rna_df.rename(columns={'Case ID': 'PATIENT_ID'}, inplace=True)
df = rna_df.merge(df, how='inner', on='PATIENT_ID')
print("tmb column nulls: ",df.isna().sum()['TMB_NONSYNONYMOUS'])
df.dropna(subset=['TMB_NONSYNONYMOUS'], inplace=True) # Ensure we only keep patients with known TMB

# --- 2. Isolate Unique Patients ---
# We must split by patient, not by row/patch
unique_patients = df[['PATIENT_ID', 'TMB_NONSYNONYMOUS']].drop_duplicates(subset=['PATIENT_ID'])
print(f"Total unique patients found: {len(unique_patients)}")

# --- 3. Stratify by Continuous TMB ---
# We temporarily group TMB into bins (quartiles) to ensure train/val/test
# all have an equal share of Low, Medium, and High TMB patients.
unique_patients['TMB_Bin'] = pd.qcut(unique_patients['TMB_NONSYNONYMOUS'], q=4, labels=False)

# --- 4. Perform the Splits ---
# Step A: Split off the Training set (70%)
train_df, temp_df = train_test_split(
    unique_patients,
    test_size=(1.0 - TRAIN_RATIO),
    random_state=RANDOM_SEED,
    stratify=unique_patients['TMB_Bin'] # Ensures balanced TMB distributions
)

# Step B: Split the remaining 30% into Validation (15%) and Test (15%)
# Since Val and Test are equal, we split temp_df 50/50
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=temp_df['TMB_Bin']
)

# --- 5. Map the Labels ---
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

# Combine back into a single master ledger
master_split_df = pd.concat([train_df, val_df, test_df])

# Clean up temporary columns
master_split_df = master_split_df[['PATIENT_ID', 'split']].sort_values(by='PATIENT_ID').reset_index(drop=True)

# --- 6. Save and Verify ---
master_split_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nMaster split saved successfully to: {OUTPUT_PATH}")

print("\n--- Split Verification ---")
print(master_split_df['split'].value_counts(normalize=True) * 100)

Loading master dataset...
tmb column nulls:  27
Total unique patients found: 350

Master split saved successfully to: /home/maro/final-projects/DSAI_305_XAI/data/processed/master_patient_splits.csv

--- Split Verification ---
split
train    69.714286
test     15.142857
val      15.142857
Name: proportion, dtype: float64
